In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

# Imports from PyTorch.
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch import nn, Tensor, device, no_grad, manual_seed
from torch import max as torch_max
from torch.utils.data import DataLoader
from torchvision.datasets.utils import download_url
import torch.optim.lr_scheduler as lr_scheduler
import torch.nn.functional as F
from copy import deepcopy

import sys
# get the path of the current file
file = os.path.abspath('')
path = os.path.join(file, '../../src')
sys.path.append(path)
print(path)


# Imports from aihwkit.
from aihwkit.nn import AnalogConv2d, AnalogLinear, AnalogSequential
from aihwkit.optim import AnalogSGD
from aihwkit.simulator.configs.configs import (
    InferenceRPUConfig,
    # remember, the InferenceRPUConfig configuration parameter is used only for inference:
    # this means that the hardware non-idealities are considered only in the forward pass,
    # while the backward and update passes are ideal.
)
from aihwkit.simulator.configs import MappingParameter
from aihwkit.simulator.configs import (
    SingleRPUConfig,
    FloatingPointRPUConfig,
    ConstantStepDevice,
    FloatingPointDevice,
)


from aihwkit.simulator.parameters import (
    WeightClipParameter,
    WeightQuantizerParameter,
)
from aihwkit.simulator.parameters.enums import WeightQuantizerType

from aihwkit.nn.conversion import convert_to_analog
from aihwkit.simulator.presets import StandardHWATrainingPreset
from aihwkit.simulator.parameters.enums import WeightClipType

from aihwkit.simulator.rpu_base import cuda



# Check device
USE_CUDA = 0
if cuda.is_compiled():
    USE_CUDA = 1

if USE_CUDA:
    torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0" if USE_CUDA else "cpu")
print("Device: ", DEVICE)


cur_dir = os.getcwd()
print("Current directory: ", cur_dir)


# Path to store datasets
PATH_DATASET = os.path.join(cur_dir, "../data/cifar10")
print("Path to store datasets: ", PATH_DATASET)



# Training parameters
SEED = 1
N_EPOCHS = 40
BATCH_SIZE =256
LEARNING_RATE = 0.05
N_CLASSES = 10

/home/ecabiati/cellar/aihwkit/sandbox/cuda/../../src
Device:  cuda:0
Current directory:  /home/ecabiati/cellar/aihwkit/sandbox/cuda
Path to store datasets:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/cifar10


In [2]:
class LambdaLayer(torch.nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(torch.nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, option="A", RPU_CONFIG=None):
        super(BasicBlock, self).__init__()
        self.conv1 = AnalogConv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False, rpu_config=RPU_CONFIG,
        )
        self.bn1 = torch.nn.BatchNorm2d(planes)
        self.conv2 = AnalogConv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False, rpu_config=RPU_CONFIG,
        )
        self.bn2 = torch.nn.BatchNorm2d(planes)

        self.shortcut = AnalogSequential()
        if stride != 1 or in_planes != planes:
            if option == "A":
                """
                For CIFAR10 ResNet paper uses option A.
                """
                self.shortcut = LambdaLayer(
                    lambda x: F.pad(
                        x[:, :, ::2, ::2],
                        (0, 0, 0, 0, planes // 4, planes // 4),
                        "constant",
                        0,
                    )
                )
            elif option == "B":
                self.shortcut = AnalogSequential(
                    AnalogConv2d(
                        in_planes,
                        self.expansion * planes,
                        kernel_size=1,
                        stride=stride,
                        bias=False,
                        rpu_config=RPU_CONFIG,
                    ),
                    torch.nn.BatchNorm2d(self.expansion * planes),
                )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


def Resnet9(channels, RPU_CONFIG, first_layer_quantization = True):
    """
    From https://github.com/matthias-wright/cifar10-resnet/
    """

    # resnet9 [56,112,224,224]
    # resnet9s [28,28,28,56]

    FIRST_LAYER_RPU_CONFIG = deepcopy(RPU_CONFIG)
    if first_layer_quantization == False:
        FIRST_LAYER_RPU_CONFIG.quantization = WeightQuantizerParameter(
            quantizer_type=WeightQuantizerType.NONE
        )

        print("No quantization for the first layer has been requested.")
        print("First layer RPU_CONFIG: ", FIRST_LAYER_RPU_CONFIG.quantization)
        print("Original RPU_CONFIG quantization parameter: ", RPU_CONFIG.quantization)


    model = AnalogSequential(
        # prep
        AnalogConv2d(
            in_channels=3,
            out_channels=channels[0],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
            rpu_config=FIRST_LAYER_RPU_CONFIG,
        ),
        torch.nn.BatchNorm2d(num_features=channels[0], momentum=0.9),
        torch.nn.ReLU(inplace=True),
        # Layer 1
        AnalogConv2d(
            in_channels=channels[0],
            out_channels=channels[1],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
            rpu_config=RPU_CONFIG,
        ),
        torch.nn.BatchNorm2d(num_features=channels[1], momentum=0.9),
        torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),
        BasicBlock(in_planes=channels[1], planes=channels[1], stride=1, RPU_CONFIG=RPU_CONFIG),
        # Layer 2
        AnalogConv2d(
            in_channels=channels[1],
            out_channels=channels[2],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
            rpu_config=RPU_CONFIG,
        ),
        torch.nn.BatchNorm2d(num_features=channels[2], momentum=0.9),
        torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),
        # Layer 3
        AnalogConv2d(
            in_channels=channels[2],
            out_channels=channels[3],
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
            rpu_config=RPU_CONFIG,
        ),
        torch.nn.BatchNorm2d(num_features=channels[3], momentum=0.9),
        torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),
        BasicBlock(in_planes=channels[3], planes=channels[3], stride=1, RPU_CONFIG=RPU_CONFIG),
        torch.nn.MaxPool2d(kernel_size=4, stride=4),
        torch.nn.Flatten(),
        AnalogLinear(in_features=channels[3], out_features=10, bias=True, rpu_config=RPU_CONFIG)
    )

    return model



def create_analog_network(RPU_CONFIG, first_layer_quantization = True):
    return Resnet9(channels=[28,28,28,56], RPU_CONFIG=RPU_CONFIG, first_layer_quantization = first_layer_quantization)




In [3]:
class EarlyStopper:
    def __init__(self, patience=7, min_delta=0.005):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss >= (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False


def load_images():
    train_set = datasets.CIFAR10(root='../data/cifar10', train=True, download=True, transform=transforms.ToTensor())
    # val_set = datasets.CIFAR10(root='../data/cifar10', train=False, download=True, transform=transforms.ToTensor())

    mean = torch.mean(torch.stack([torch.mean(image, dim=(1, 2)) for image, _ in train_set]), dim=0)
    std = torch.std(torch.stack([torch.std(image, dim=(1, 2)) for image, _ in train_set]), dim=0)
    print(f"Normalization data: ({mean},{std})")
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

    train_set = datasets.CIFAR10(PATH_DATASET, train=True, download=True, transform=transform)
    val_set = datasets.CIFAR10(PATH_DATASET, train=False, download=True, transform=transform)
    train_data = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    validation_data = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    return train_data, validation_data


def create_sgd_optimizer(model, learning_rate, gamma = 0.9):
    """Create the analog-aware optimizer.

    Args:
        model (nn.Module): model to be trained
        learning_rate (float): global parameter to define learning rate
    Returns:
        Optimizer: optimizer
    """
    optimizer = AnalogSGD(model.parameters(), lr=learning_rate)
    scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    optimizer.regroup_param_groups(model)

    return optimizer , scheduler


def train_step(train_data, model, criterion, optimizer, scheduler):
    """Train network.

    Args:
        train_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer

    Returns:
        nn.Module, Optimizer, float: model, optimizer, and epoch loss
    """
    total_loss = 0

    model.train()

    for images, labels in train_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()

        # Add training Tensor to the model (input).
        output = model(images)
        loss = criterion(output, labels)

        # Run training (backward propagation).
        loss.backward()

        # Optimize weights.
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    epoch_loss = total_loss / len(train_data.dataset)
    #scheduler.step(epoch_loss)

    return model, optimizer, epoch_loss


def test_evaluation(validation_data, model, criterion):
    """Test trained network

    Args:
        validation_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss

    Returns:
        nn.Module, float, float, float: model, test epoch loss, test error, and test accuracy
    """
    total_loss = 0
    predicted_ok = 0
    total_images = 0

    model.eval()

    for images, labels in validation_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        pred = model(images)
        loss = criterion(pred, labels)
        total_loss += loss.item() * images.size(0)

        _, predicted = torch_max(pred.data, 1)
        total_images += labels.size(0)
        predicted_ok += (predicted == labels).sum().item()
        accuracy = predicted_ok / total_images * 100
        error = (1 - predicted_ok / total_images) * 100

    epoch_loss = total_loss / len(validation_data.dataset)

    return model, epoch_loss, error, accuracy


def training_loop(model, criterion, optimizer, scheduler, train_data, validation_data, epochs, RESULTS,  print_every=1):
    """Training loop.

    Args:
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer
        train_data (DataLoader): Validation set to perform the evaluation
        validation_data (DataLoader): Validation set to perform the evaluation
        epochs (int): global parameter to define epochs number
        print_every (int): defines how many times to print training progress

    Returns:
        nn.Module, Optimizer, Tuple: model, optimizer, and a tuple of
            lists of train losses, validation losses, and test error

    """
    train_losses = []
    valid_losses = []
    test_error = []
    early_stopper = EarlyStopper(patience=7, min_delta=0.1)

    # Train model
    for epoch in range(0, epochs):
        # Train_step
        model, optimizer, train_loss = train_step(train_data, model, criterion, optimizer, scheduler)
        train_losses.append(train_loss)

        if epoch % print_every == (print_every - 1):
            # Validate_step
            with no_grad():
                model, valid_loss, error, accuracy = test_evaluation(
                    validation_data, model, criterion
                )
                valid_losses.append(valid_loss)
                test_error.append(error)

            

            print(
                f"{datetime.now().time().replace(microsecond=0)} --- "
                f"Epoch: {epoch}\t"
                f"lr: {optimizer.param_groups[0]["lr"]:.4f}\t"
                f"Train loss: {train_loss:.4f}\t"
                f"Valid loss: {valid_loss:.4f}\t"
                f"Test error: {error:.2f}%\t"
                f"Test accuracy: {accuracy:.2f}%\t"
            )
        
        # Early stopping
        if early_stopper.early_stop(valid_loss):             
            break

    # Save results and plot figures
    np.savetxt(os.path.join(RESULTS, "Test_error.csv"), test_error, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Train_Losses.csv"), train_losses, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Valid_Losses.csv"), valid_losses, delimiter=",")
    plot_results(train_losses, valid_losses, test_error, RESULTS)

    return model, optimizer, (train_losses, valid_losses, test_error)


def plot_results(train_losses, valid_losses, test_error, RESULTS):
    """Plot results.

    Args:
        train_losses (List): training losses as calculated in the training_loop
        valid_losses (List): validation losses as calculated in the training_loop
        test_error (List): test error as calculated in the training_loop
    """
    fig = plt.plot(train_losses, "r-s", valid_losses, "b-o")
    plt.title("aihwkit Resnet9")
    plt.legend(fig[:2], ["Training Losses", "Validation Losses"])
    plt.xlabel("Epoch number")
    plt.ylabel("Loss [A.U.]")
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_losses.png"))
    plt.close()

    fig = plt.plot(test_error, "r-s")
    plt.title("aihwkit Resnet9")
    plt.legend(fig[:1], ["Test Error"])
    plt.xlabel("Epoch number")
    plt.ylabel("Test Error [%]")
    plt.yscale("log")
    plt.ylim((5e-1, 1e2))
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_error.png"))
    plt.close()


In [4]:
def main_unquantized():
    """Train a PyTorch CNN analog model with the MNIST dataset."""
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    # Path to store results
    RESULTS = os.path.join(cur_dir, "resnet9_results")
    print("Path to store results: ", RESULTS)
    os.makedirs(RESULTS, exist_ok=True)
    manual_seed(SEED)

    # Load datasets.
    train_data, validation_data = load_images()

    # Select RPU_CONFIG
    # RPU_CONFIG = StandardHWATrainingPreset()
    # RPU_CONFIG.clipping = WeightClipParameter(
    #     type = WeightClipType.NONE
    # )

    # RPU_CONFIG = InferenceRPUConfig()
    RPU_CONFIG = FloatingPointRPUConfig()

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG)
    if USE_CUDA:
        model.cuda()
    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started Resnet9 Example")

    optimizer, scheduler = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, scheduler ,train_data, validation_data, N_EPOCHS, RESULTS
    )
    
    # Save the model to .th file
    torch.save(model.state_dict(), os.path.join(RESULTS, "resnet9.th"))

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed Resnet9 Example")

    

def main_quantized():
    """Train a PyTorch CNN analog model with the MNIST dataset."""
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    # Path to store results
    RESULTS = os.path.join(cur_dir, "resnet9_quantized_results")
    print("Path to store results: ", RESULTS)

    os.makedirs(RESULTS, exist_ok=True)
    manual_seed(SEED)

    # Load datasets.
    train_data, validation_data = load_images()

    # Select RPU_CONFIG
    # RPU_CONFIG = StandardHWATrainingPreset()
    # RPU_CONFIG.clipping = WeightClipParameter(
    #     type = WeightClipType.NONE
    # )

    # RPU_CONFIG = InferenceRPUConfig()
    RPU_CONFIG = FloatingPointRPUConfig()

    RPU_CONFIG.quantization = WeightQuantizerParameter(
        resolution = 0.3,
        #amax_channelwise= True,
        eps= 0.25,
        levels = 3,
        method = "percentile",
        use_forward=True,
    )

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG, first_layer_quantization=False)
    if USE_CUDA:
        model.cuda()
    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started Resnet9 Example")

    optimizer, scheduler = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, scheduler ,train_data, validation_data, N_EPOCHS, RESULTS
    )

    # Save the model to .th file
    torch.save(model.state_dict(), os.path.join(RESULTS, "resnet9_quant.th"))

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed Resnet9 Example")

In [17]:
main_unquantized()

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/resnet9_results
Files already downloaded and verified
Files already downloaded and verified
Normalization data: (tensor([0.4914, 0.4822, 0.4465]),tensor([0.0606, 0.0612, 0.0677]))
Files already downloaded and verified
Files already downloaded and verified
AnalogSequential(
  (0): AnalogConv2d(
    3, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(28,27))
  )
  (1): BatchNorm2d(28, eps=1e-05, momentum=0.9, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): AnalogConv2d(
    28, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(28,252))
  )
  (4): BatchNorm2d(28, eps=1e-05, momentum=0.9, affine=True, track_running_stats=True)
  (5): ReLU(inplace=True)
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=

In [18]:
main_quantized()

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/resnet9_quantized_results
Files already downloaded and verified
Files already downloaded and verified
Normalization data: (tensor([0.4914, 0.4822, 0.4465]),tensor([0.0606, 0.0612, 0.0677]))
Files already downloaded and verified
Files already downloaded and verified
No quantization for the first layer has been requested.
First layer RPU_CONFIG:  WeightQuantizerParameter(
    amax_values=[
        0.0
    ],
    quantizer_type=WeightQuantizerType.NONE
)
Original RPU_CONFIG quantization parameter:  WeightQuantizerParameter(
    resolution=0.3,
    amax_values=[
        0.0
    ],
    eps=0.25,
    levels=3,
    quantizer_type=WeightQuantizerType.UNIFORM_SYMMETRIC,
    use_forward=True
)
AnalogSequential(
  (0): AnalogConv2d(
    3, 28, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(28,27))
  )
  (1): BatchNorm2d(28, eps=1e-05, momentum=0

In [8]:
from aihwkit.nn.conversion import convert_to_analog

# Load datasets.
_, validation_data = load_images()

criterion = nn.CrossEntropyLoss()

# from shared import resnet9s, CustomDefinedPreset
# rpu_config = CustomDefinedPreset()

# RESULTS_NON_QUANTIZED = cur_dir + "/../resnet"
# state_dict = torch.load(os.path.join(RESULTS_NON_QUANTIZED, "resnet9s.th"), map_location=DEVICE)
# aihwkit_model = resnet9s().to(DEVICE)
# aihwkit_model.load_state_dict(state_dict["model_state_dict"], strict=True)
# aihwkit_model = convert_to_analog(aihwkit_model, rpu_config)

rpu_config = FloatingPointRPUConfig()

# Standard model
RESULTS_NON_QUANTIZED = cur_dir + "/resnet9_results"
state_dict = torch.load(os.path.join(RESULTS_NON_QUANTIZED, "resnet9.th"), map_location=DEVICE)
non_quantized_model = create_analog_network(rpu_config).to(DEVICE)
non_quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)

# QA model
RESULTS_QUANTIZED = cur_dir + "/resnet9_quantized_results"
state_dict = torch.load(os.path.join(RESULTS_QUANTIZED, "resnet9_quant.th"), map_location=DEVICE)
quantized_model = create_analog_network(rpu_config).to(DEVICE)
quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)

# apply quantization on the QA model, the same for which it was trained
rpu_config.quantization = WeightQuantizerParameter(
        resolution = 0.3,
        #amax_channelwise= True,
        eps= 0.25,
        levels = 3,
        method = "percentile"
    )

#non_quantized_model = convert_to_analog(non_quantized_model, rpu_config)
quantized_model = convert_to_analog(quantized_model, rpu_config, apply_quant_on_first=False)


# Evaluate the models
non_quantized_model.eval()
quantized_model.eval()

_, _, _, accuracy_non_quantized = test_evaluation(validation_data, non_quantized_model, criterion)
_, _, _, accuracy_quantized = test_evaluation(validation_data, quantized_model, criterion)

print(f"Accuracy of the non-quantized model: {accuracy_non_quantized:.2f}%")
print(f"Accuracy of the quantized model: {accuracy_quantized:.2f}%")


# Plot the weights of the first layer
src = os.getcwd()
src = os.path.abspath(os.path.join(src, '../src'))
sys.path.append(src)

import plotting as pl

pl.generate_moving_hist(non_quantized_model, title = "Non-quantized model", file_name = RESULTS_NON_QUANTIZED + "/weights_hist_non_quantized.gif", range = (-1, 1), top = None, split_by_rows=False)
pl.generate_moving_hist(quantized_model, title = "Quantized model", file_name = RESULTS_QUANTIZED + "/weights_hist_quantized.gif", range = (-1, 1), top = None, split_by_rows=False)
# pl.generate_moving_hist(aihwkit_model, title = "AIHWKit model", file_name = RESULTS_NON_QUANTIZED + "/weights_hist_aihwkit.gif", range = (-1, 1), top = None, split_by_rows=False)






Files already downloaded and verified
Normalization data: (tensor([0.4914, 0.4822, 0.4465]),tensor([0.0606, 0.0612, 0.0677]))
Files already downloaded and verified
Files already downloaded and verified


































Accuracy of the non-quantized model: 76.32%
Accuracy of the quantized model: 56.27%


<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>